In [1]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("Working in:", os.getcwd())

Working in: /Users/jamesfamodimu/Documents/energy-efficiency


In [9]:
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.multitest import multipletests
from sklearn.model_selection import cross_val_predict

for d in ["figures", "outputs"]:
    Path(d).mkdir(exist_ok=True)
pd.set_option("display.width", 110); pd.set_option("display.max_columns", 20)
print("Setup is complete")

Setup is complete


In [15]:
import joblib
from src.data import load_data, split_data, SEED, TARGETS, NUM, CAT, THRESH

random_number_generator = np.random.default_rng(SEED)
df = load_data()
X_tr, X_te, Y_tr, Y_te = split_data(df)

features_all = df[NUM + CAT]
targets_all = df[TARGETS]

final_models = joblib.load("model/tuned_models.joblib")
logistic_regression_pipeline = joblib.load("model/logit.joblib")

print(X_tr.shape, X_te.shape, "| models:", list(final_models))


(614, 7) (154, 7) | models: ['heating_load', 'cooling_load']


In [16]:
from sklearn.model_selection import KFold, cross_val_predict

def calculate_conformal_quantile(pipeline, features, target_values):
    out_of_fold_predictions = cross_val_predict(
        estimator=pipeline, 
        X=features, 
        y=target_values, 
        cv=KFold(n_splits=10, shuffle=True, random_state=SEED), 
        n_jobs=-1
    )
    absolute_residuals = np.abs(target_values - out_of_fold_predictions)
    sample_size = len(absolute_residuals)
    
    conformal_quantile = np.quantile(
        absolute_residuals, 
        min(1.0, np.ceil((sample_size + 1) * 0.95) / sample_size)
    )
    return conformal_quantile, (target_values - out_of_fold_predictions).values

conformal_quantiles = {}
prediction_residuals = {}
conformal_records = []

for target_variable in TARGETS:
    target_quantile, target_residuals = calculate_conformal_quantile(
        pipeline=final_models[target_variable], 
        features=X_tr, 
        target_values=Y_tr[target_variable]
    )
    conformal_quantiles[target_variable] = target_quantile
    prediction_residuals[target_variable] = target_residuals
    
    test_predictions = final_models[target_variable].predict(X_te)
    test_coverage_rate = (np.abs(Y_te[target_variable] - test_predictions) <= target_quantile).mean()
    
    conformal_records.append([target_variable, target_quantile, test_coverage_rate])

conformal_results_table = pd.DataFrame(
    conformal_records, 
    columns=["target", "halfwidth_95", "test_coverage"]
)
conformal_results_table.to_csv("outputs/conformal.csv", index=False)

print(conformal_results_table.round(3))


         target  halfwidth_95  test_coverage
0  heating_load         0.804          0.987
1  cooling_load         1.882          0.994
